# Metro-ASR — Gradio Web Demo

Run the same Gradio UI documented in the [README's Serving section]
(https://github.com/MohammedAly22/metro-asr#serving):
- File upload & microphone recording, with a decoding-method selector and live metrics
- Live microphone streaming tab
- Dark, RTL-aware UI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/metro-asr/blob/main/examples/gradio_app.ipynb)


## Option A — run the full app from the repo

This is `app.py` exactly as it ships in the repository. Two environment variables matter
in a hosted notebook, since Colab has no direct route to `localhost`:

- `METRO_MODEL=small` — the repo's `checkpoints/` directory doesn't exist in a fresh clone,
  so point the app at the HuggingFace size alias instead; it downloads on first launch.
- `METRO_SHARE=true` — asks Gradio for a public `https://*.gradio.live` tunnel URL instead
  of binding only to the container's internal `localhost`.

Beam search + the language model add real weight (a 6 GB download, and the `[lm]` extra) for what's meant to be a quick UI demo, so it's off by default here — see the commented-out `METRO_LM` line in the launch cell to turn it on.

In [ ]:
!pip install -q "metro-asr[demo]"
!git clone -q https://github.com/MohammedAly22/metro-asr.git
%cd metro-asr


In [ ]:
%env METRO_MODEL=small
%env METRO_SHARE=true
# app.py's own default is METRO_LM=auto, which downloads and uses the 6 GB general
# language model — see the note above the install cell for why that needs an extra
# numpy step. Setting it empty here keeps this a fast, greedy-only demo; change it to
# auto (and apply the numpy fix above) to enable beam search too.
%env METRO_LM=
!python app.py


## Option B — minimal inline demo

No repo clone, no `app.py` — a small self-contained Gradio interface built directly on
`MetroASREngine`, useful if you just want a quick shareable link for a single model.

In [ ]:
!pip install -q "metro-asr[demo]"

from metro_asr import MetroASREngine
import gradio as gr

engine = MetroASREngine.from_pretrained("small")

def transcribe(audio):
    if audio is None:
        return ""
    return engine.transcribe(audio).text

demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(type="filepath", sources=["upload", "microphone"]),
    outputs=gr.Textbox(label="Transcription", rtl=True),
    title="Metro-ASR",
    description="Egyptian Arabic + Code-Switching Speech Recognition",
)

demo.launch(share=True)


## Next steps

- **[Streaming & REST API](streaming_server.ipynb)** — the server behind this UI, callable
  directly
- **[Quick start](quick_start.ipynb)** — the underlying `MetroASREngine` API this app wraps
